# install requirements:

In [0]:
%pip install -r /Volumes/workspace/default/real-estate/requirements.txt

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


# restart kernel to apply requirements:

In [0]:
%restart_python

# Run the imot.bg sitemap discovery 

## fiter by slug required_slug_keywords=["prodava-kashta", "varna"]

In [0]:
import sys

sys.path.append("/Volumes/workspace/default/real-estate/code")

import imot_bg_sitemap_discovery

houses_for_sale_in_varna = imot_bg_sitemap_discovery.discover_listing_id_url_pairs(
    required_slug_keywords=["prodava-kashta", "varna"]
)

print(f"total matched: {len(houses_for_sale_in_varna)}")

imot_bg_sitemap_discovery.save_pairs_to_file(houses_for_sale_in_varna)

found 40 listing sitemap files
  [1/40] listings-1.xml.gz: 5880 urls, 5 matched
  [2/40] listings-10.xml.gz: 5244 urls, 49 matched
  [3/40] listings-11.xml.gz: 5173 urls, 38 matched
  [4/40] listings-12.xml.gz: 5097 urls, 37 matched
  [5/40] listings-13.xml.gz: 5146 urls, 26 matched
  [6/40] listings-14.xml.gz: 5131 urls, 34 matched
  [7/40] listings-15.xml.gz: 4954 urls, 50 matched
  [8/40] listings-16.xml.gz: 5076 urls, 28 matched
  [9/40] listings-17.xml.gz: 5207 urls, 40 matched
  [10/40] listings-18.xml.gz: 5353 urls, 38 matched
  [11/40] listings-19.xml.gz: 5511 urls, 11 matched
  [12/40] listings-2.xml.gz: 5576 urls, 7 matched
  [13/40] listings-20.xml.gz: 5645 urls, 8 matched
  [14/40] listings-21.xml.gz: 5358 urls, 13 matched
  [15/40] listings-22.xml.gz: 5549 urls, 17 matched
  [16/40] listings-23.xml.gz: 5602 urls, 28 matched
  [17/40] listings-24.xml.gz: 5525 urls, 55 matched
  [18/40] listings-25.xml.gz: 5623 urls, 119 matched
  [19/40] listings-26.xml.gz: 5619 urls, 136 m

# Run the scraper based on the sitemaps
## expected time 1–1.5 hours for a full run of 2510 listing URLs and that is just for Varna houses for sale

In [0]:
import sys

sys.path.append("/Volumes/workspace/default/real-estate/code")

import imot_bg_sitemap_discovery
import imot_bg_scraper

listing_id_url_pairs = imot_bg_sitemap_discovery.load_pairs_from_file()

imot_bg_scraper.scrape_listing_urls(listing_id_url_pairs)

got 2514 listing URLs, 2504 existing listings
OK 1j172561998634146 249900 EUR
OK 1j177124626112887 509999 EUR
OK 1j178086819491705 79000 EUR
OK 1j177133492670833 149500 EUR
OK 1j175432140423702 170000 EUR
OK 1j178983385754308 550000 EUR
OK 1j176613845025461 420000 EUR
OK 1j178999569579811 219900 EUR
OK 1j178791132803371 126000 EUR
OK 1j178947967547141 219001 EUR
OK 1j173928195066297 39999 EUR
OK 1j176861894570750 359000 EUR
OK 1j178039090680339 299900 EUR
OK 1j177392644413626 271920 EUR
OK 1j177392645285001 274392 EUR
OK 1j177305262150279 50000 EUR
OK 1j177392644863056 271920 EUR
OK 1j176598117674503 679000 EUR
OK 1j164131392570941 55000 EUR
OK 1j176372588305811 150000 EUR
NEW 1j179025086355676 79000 EUR
OK 1j177946522712803 395000 EUR
OK 1j178697742472941 585000 EUR
OK 1j166020728130526 67990 EUR
OK 1j178782576465436 90000 EUR
OK 1j178765852488731 55000 EUR
NEW 1j179015081656700 450000 EUR
OK 1j178428324225626 При запитване EUR
OK 1j178998969047956 350000 EUR
PRICE CHANGE 1j1787658569

# check some sample data

In [0]:
import json

detail_path = "/Volumes/workspace/default/real-estate/raw/detail_snapshot.jsonl"
history_path = "/Volumes/workspace/default/real-estate/raw/price_history.jsonl"

# Count current listings
with open(detail_path, encoding="utf-8") as f:
    detail_rows = [json.loads(line) for line in f]

# Count price history records
with open(history_path, encoding="utf-8") as f:
    history_rows = [json.loads(line) for line in f]

print("Current listings:", len(detail_rows))
print("Price history records:", len(history_rows))

print("\nExample listing:")
print(json.dumps(detail_rows[0], ensure_ascii=False, indent=2))

print("\nExample price history:")
print(json.dumps(history_rows[0], ensure_ascii=False, indent=2))

Current listings: 2514
Price history records: 2176

Example listing:
{
  "listing_id": "1j172561998634146",
  "url": "https://www.imot.bg/obiava-1j172561998634146-prodava-kashta-oblast-varna-gr-byala-tsentar",
  "price": 249900,
  "currency": "EUR",
  "agency_name": "МД ОБЗОР ЕООД",
  "location_text": "област Варна, гр. Бяла ЦЕНТЪР",
  "property_parameters": {
    "Площ": "221 m2",
    "Двор": "774 m2",
    "Етаж": "2",
    "Строителство": "Тухла,"
  },
  "description": "ЕКСКЛУЗИВНА ОФЕРТА! БЕЗ КОМИСИОННА ОТ КУПУВАЧА! Имаме удоволствието да Ви предложим за продажба новоизградена луксозна двуетажна къща   Американска мечта  в град Бяла, която си заслужава да бъде разгледана! Разположена е в центъра на красивия морски курорт с лице към реновирана улица, в близост до новата автогара, магазини, училище. Къщата е с акт 14 от 2024г. и се продава  на тапа  в перфектно състояние и готова да посрещне новите си собственици, които да превърнат сградата в собствен уютен дом според своите предпочит

# Create Bronze tables for listings and price history

In [0]:
from pyspark.sql import functions as F

detail_path = "/Volumes/workspace/default/real-estate/raw/detail_snapshot.jsonl"

df = spark.read.json(detail_path)

df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.bronze_imot_listings")

price_history_path = "/Volumes/workspace/default/real-estate/raw/price_history.jsonl"

price_history_df = spark.read.json(price_history_path)

price_history_df = price_history_df.withColumn(
    "observed_at",
    F.to_timestamp("observed_at")
)

price_history_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.bronze_imot_price_history")


display(df)


agency_name,currency,description,feature_tags,listing_id,listing_info_text,location_text,price,property_parameters,url
МД ОБЗОР ЕООД,EUR,"ЕКСКЛУЗИВНА ОФЕРТА! БЕЗ КОМИСИОННА ОТ КУПУВАЧА! Имаме удоволствието да Ви предложим за продажба новоизградена луксозна двуетажна къща Американска мечта в град Бяла, която си заслужава да бъде разгледана! Разположена е в центъра на красивия морски курорт с лице към реновирана улица, в близост до новата автогара, магазини, училище. Къщата е с акт 14 от 2024г. и се продава на тапа в перфектно състояние и готова да посрещне новите си собственици, които да превърнат сградата в собствен уютен дом според своите предпочитания! Изпълнени: луксозна външна дограма, електро и ВиК инсталации, мазилка, замазка, инсталация за отопление и охлаждане за термопомпа с изводи за конвертори, автоматична гаражна врата, окабеляване за интернет и телевизия; излят фундамент за плътна ограда на парцела. В северната част на парцела има възможност новите собственици да изградят малък басейн, външен бар и други съоръжение за създаване на приятна атмосфера. Сградата е присъединена към водопроводната и канализационна система на града. Открита е партида за електричество. Перфектната къща за целогодишно обитаване. На новите собственици ще бъдат на разположение: Първи жилищен етаж с гараж на площ от 130.80кв.м., състоящ се от функционален гараж 25кв.м. със складово помещение /бойлерно/ 6.14кв.м. с директен достъп към дворното пространство и топла връзка към жилищната част, която е обособена като входно антре 7.50кв.м., стая за гости 10.10кв.м., баня 3.37кв.м. и просторна панорамна всекидневна /трапезария, кухня/ 48кв.м. с френски прозорци и директен достъп към покрита веранда и голям двор. Посредством вътрешно стълбище е осигурен достъп до Втори жилищен етаж 80кв.м. с разпределение: родителска спалня 16кв.м. със санитарен възел 5кв.м. и гардеробна 6кв.м. и огромна панорамна тераса; детска спалня с отделна баня; обособено е и перално помещение на площ от 4.40кв.м. Към момента се изпълнява външна изолация, след което цената ще се актуализира! Вашата мечта може да стане реалност!","List(Тухла, С гараж, С паркинг, Интернет връзка)",1j172561998634146,"Коригирана в 12:17 на 18 септември, 2026 год. Обявата е посетена 5468 пъти.","област Варна, гр. Бяла ЦЕНТЪР",249900,"List(null, 774 m2, 2, 221 m2, Тухла,, null)",https://www.imot.bg/obiava-1j172561998634146-prodava-kashta-oblast-varna-gr-byala-tsentar
МОДА ИМОТИ,EUR,"Стара цена: 550 000 / 1 075 706 лв. Нова цена: 509 999 / 997 472 лв. Отстъпка: 40 001 / 78 235 лв. Продава се самостоятелна къща в един от най-престижните райони на Варна кв. Св. Никола. Уникална локация с панорамна и напълно открита морска гледка, която не може да бъде закрита от бъдещо строителство. Застроена площ: 265 кв.м Парцел: 858 кв.м Имотът се намира на главната улица арх. Манол Йорданов, разделяща местностите Св. Никола и Долна Трака. Къщата е в отлично състояние, с функционално и удобно разпределение. Разпределение: Партер: Просторна дневна 55 кв.м с обособени кухненска, трапезарна и холна зона Санитарен възел Първи полуетаж: Спалня (подходяща за кабинет) Голяма гардеробна Баня с тоалетна Втори етаж: Две спални с морска панорама Баня с тоалетна Следващ полуетаж: Четвърта спалня със собствен санитарен възел Покривен етаж: Затворена зона с PVC дограма, предназначена за релакс Невероятна, напълно открита морска панорама Предимства: Топ локация в една от най-предпочитаните вилни зони на Варна Редовни документи Възможност за незабавно ползване Голям терасиран парцел с аранжирана растителност Обособени зони за стопанска дейност 4 паркоместа Къщата се продава обзаведена Панорамна къща с морска гледка на изключително атрактивна цена. Защо да изберете нас? 20 години опит на пазара на недвижими имоти; качество на обслужването и честност; ЛИЦЕНЗИРАН БРОКЕР нашият управител има магистърска степен по 'Икономика на недвижимите имоти' и е изучавал вещно право в университета, както и 'Основи на правото' в гимназията в специал

# test: query the delta bronze tables

In [0]:
%sql
SELECT *
FROM workspace.default.bronze_imot_listings
LIMIT 20;



agency_name,currency,description,feature_tags,listing_id,listing_info_text,location_text,price,property_parameters,url
МД ОБЗОР ЕООД,EUR,"ЕКСКЛУЗИВНА ОФЕРТА! БЕЗ КОМИСИОННА ОТ КУПУВАЧА! Имаме удоволствието да Ви предложим за продажба новоизградена луксозна двуетажна къща Американска мечта в град Бяла, която си заслужава да бъде разгледана! Разположена е в центъра на красивия морски курорт с лице към реновирана улица, в близост до новата автогара, магазини, училище. Къщата е с акт 14 от 2024г. и се продава на тапа в перфектно състояние и готова да посрещне новите си собственици, които да превърнат сградата в собствен уютен дом според своите предпочитания! Изпълнени: луксозна външна дограма, електро и ВиК инсталации, мазилка, замазка, инсталация за отопление и охлаждане за термопомпа с изводи за конвертори, автоматична гаражна врата, окабеляване за интернет и телевизия; излят фундамент за плътна ограда на парцела. В северната част на парцела има възможност новите собственици да изградят малък басейн, външен бар и други съоръжение за създаване на приятна атмосфера. Сградата е присъединена към водопроводната и канализационна система на града. Открита е партида за електричество. Перфектната къща за целогодишно обитаване. На новите собственици ще бъдат на разположение: Първи жилищен етаж с гараж на площ от 130.80кв.м., състоящ се от функционален гараж 25кв.м. със складово помещение /бойлерно/ 6.14кв.м. с директен достъп към дворното пространство и топла връзка към жилищната част, която е обособена като входно антре 7.50кв.м., стая за гости 10.10кв.м., баня 3.37кв.м. и просторна панорамна всекидневна /трапезария, кухня/ 48кв.м. с френски прозорци и директен достъп към покрита веранда и голям двор. Посредством вътрешно стълбище е осигурен достъп до Втори жилищен етаж 80кв.м. с разпределение: родителска спалня 16кв.м. със санитарен възел 5кв.м. и гардеробна 6кв.м. и огромна панорамна тераса; детска спалня с отделна баня; обособено е и перално помещение на площ от 4.40кв.м. Към момента се изпълнява външна изолация, след което цената ще се актуализира! Вашата мечта може да стане реалност!","List(Тухла, С гараж, С паркинг, Интернет връзка)",1j172561998634146,"Коригирана в 12:17 на 18 септември, 2026 год. Обявата е посетена 5468 пъти.","област Варна, гр. Бяла ЦЕНТЪР",249900,"List(null, 774 m2, 2, 221 m2, Тухла,, null)",https://www.imot.bg/obiava-1j172561998634146-prodava-kashta-oblast-varna-gr-byala-tsentar
МОДА ИМОТИ,EUR,"Стара цена: 550 000 / 1 075 706 лв. Нова цена: 509 999 / 997 472 лв. Отстъпка: 40 001 / 78 235 лв. Продава се самостоятелна къща в един от най-престижните райони на Варна кв. Св. Никола. Уникална локация с панорамна и напълно открита морска гледка, която не може да бъде закрита от бъдещо строителство. Застроена площ: 265 кв.м Парцел: 858 кв.м Имотът се намира на главната улица арх. Манол Йорданов, разделяща местностите Св. Никола и Долна Трака. Къщата е в отлично състояние, с функционално и удобно разпределение. Разпределение: Партер: Просторна дневна 55 кв.м с обособени кухненска, трапезарна и холна зона Санитарен възел Първи полуетаж: Спалня (подходяща за кабинет) Голяма гардеробна Баня с тоалетна Втори етаж: Две спални с морска панорама Баня с тоалетна Следващ полуетаж: Четвърта спалня със собствен санитарен възел Покривен етаж: Затворена зона с PVC дограма, предназначена за релакс Невероятна, напълно открита морска панорама Предимства: Топ локация в една от най-предпочитаните вилни зони на Варна Редовни документи Възможност за незабавно ползване Голям терасиран парцел с аранжирана растителност Обособени зони за стопанска дейност 4 паркоместа Къщата се продава обзаведена Панорамна къща с морска гледка на изключително атрактивна цена. Защо да изберете нас? 20 години опит на пазара на недвижими имоти; качество на обслужването и честност; ЛИЦЕНЗИРАН БРОКЕР нашият управител има магистърска степен по 'Икономика на недвижимите имоти' и е изучавал вещно право в университета, както и 'Основи на правото' в гимназията в специал

In [0]:
%sql
SELECT *
FROM workspace.default.bronze_imot_price_history
ORDER BY observed_at DESC
LIMIT 20;

currency,listing_id,observed_at,price,url
EUR,1j179024844520508,2026-09-24T15:40:54.000Z,62000,https://www.imot.bg/obiava-1j179024844520508-prodava-kashta-oblast-varna-s-vetrino
EUR,1j179024488764899,2026-09-24T15:40:33.000Z,370000,https://www.imot.bg/obiava-1j179024488764899-prodava-kashta-grad-varna-m-t-manastirski-rid
EUR,1j179024534631420,2026-09-24T15:40:02.000Z,185000,https://www.imot.bg/obiava-1j179024534631420-prodava-kashta-oblast-varna-s-banovo
EUR,1j179024074089475,2026-09-24T15:39:02.000Z,15000,https://www.imot.bg/obiava-1j179024074089475-prodava-kashta-oblast-varna-s-sava
EUR,1j175611408436493,2026-09-24T15:38:56.000Z,775000,https://www.imot.bg/obiava-1j175611408436493-prodava-kashta-grad-varna-m-t-alen-mak
EUR,1j179015004347177,2026-09-24T15:37:51.000Z,1111000,https://www.imot.bg/obiava-1j179015004347177-prodava-kashta-grad-varna-asparuhovo
EUR,1j179023813698544,2026-09-24T15:37:26.000Z,375000,https://www.imot.bg/obiava-1j179023813698544-prodava-kashta-grad-varna-m-t-dolna-traka
EUR,1j179007546924339,2026-09-24T15:37:10.000Z,130000,https://www.imot.bg/obiava-1j179007546924339-prodava-kashta-oblast-varna-s-goren-chiflik
EUR,1j179014722062312,2026-09-24T15:37:01.000Z,219900,https://www.imot.bg/obiava-1j179014722062312-prodava-kashta-oblast-varna-gr-provadiya
EUR,1j179006981986647,2026-09-24T15:36:49.000Z,119900,https://www.imot.bg/obiava-1j179006981986647-prodava-kashta-oblast-varna-s-avren


# Bronze → Silver transformation (listings)

Bronze contains the scraped data in a raw, mostly source-like format.
In Silver, the data is cleaned, standardized, and converted into analysis-ready columns.

Main changes:

- Convert price and property measurements from strings to numeric values
- Extract area, plot area, and floor from raw text
- Convert gas and district heating values into usable boolean fields
- Standardize construction type
- Split location information into region, location type, and location name
- Classify locations such as villages, towns, localities, resorts, and neighborhoods
- Extract listing edit date and view count from listing information
- Calculate price per square meter
- Remove temporary raw/helper columns used during transformation

The goal of Silver is to preserve the useful information from Bronze while making the data consistent and easy to use for analysis and Gold tables.

In [0]:
from pyspark.sql import functions as F

# Load Bronze tables

bronze_listings = spark.table(
    "workspace.default.bronze_imot_listings"
)

bronze_price_history = spark.table(
    "workspace.default.bronze_imot_price_history"
)

# Build Silver Listings

silver_listings = (
    bronze_listings

    .select(
        "listing_id",
        "url",
        "agency_name",
        "currency",

        # Convert invalid/non-numeric prices to NULL
        F.expr("try_cast(price AS double)").alias("price"),

        "location_text",
        "description",
        "feature_tags",
        "listing_info_text",

        F.col("property_parameters.Площ").alias("area_raw"),
        F.col("property_parameters.Двор").alias("plot_area_raw"),
        F.col("property_parameters.Етаж").alias("floor_raw"),
        F.col("property_parameters.Газ").alias("gas_raw"),
        F.col("property_parameters.ТEЦ").alias("district_heating_raw"),
        F.col("property_parameters.Строителство").alias("construction_raw")
    )

    # Convert property area to numeric square metres
    .withColumn(
        "area_sqm",
        F.regexp_extract(
            "area_raw",
            r"([0-9]+(?:[.,][0-9]+)?)",
            1
        ).cast("double")
    )

    .withColumn(
        "plot_area_sqm",
        F.regexp_extract(
            "plot_area_raw",
            r"([0-9]+(?:[.,][0-9]+)?)",
            1
        ).cast("double")
    )

    # Extract the first number from the floor
    #
    # 3       -> 3
    # 3 от 3  -> 3
    # 2 от 3  -> 2
    .withColumn(
        "floor",
        F.regexp_extract(
            "floor_raw",
            r"^(\d+)",
            1
        ).cast("integer")
    )

    # Convert gas information to boolean
    #
    # ДА -> true
    # НЕ -> false
    # Other values -> NULL
    .withColumn(
        "has_gas",
        F.when(
            F.upper(F.trim("gas_raw")) == "ДА",
            True
        )
        .when(
            F.upper(F.trim("gas_raw")) == "НЕ",
            False
        )
        .otherwise(None)
    )

    # Convert district heating information to boolean
    #
    # ДА -> true
    # НЕ -> false
    # Other values -> NULL
    .withColumn(
        "has_district_heating",
        F.when(
            F.upper(F.trim("district_heating_raw")) == "ДА",
            True
        )
        .when(
            F.upper(F.trim("district_heating_raw")) == "НЕ",
            False
        )
        .otherwise(None)
    )

    # Preserve the actual heating type
    #
    # Example:
    # ДА
    # НЕ
    # Лок.отопл.
    .withColumn(
        "district_heating_type",
        F.trim("district_heating_raw")
    )

    # Clean construction type
    #
    # Тухла, -> Тухла
    .withColumn(
        "construction_type",
        F.trim(
            F.regexp_replace(
                "construction_raw",
                r",+$",
                ""
            )
        )
    )

    # Extract region from location
    #
    # област Варна -> Варна
    # град Варна   -> Варна
    .withColumn(
        "region",
        F.regexp_extract(
            "location_text",
            r"^(?:област|град)\s+([^,]+)",
            1
        )
    )

    # Extract location type
    #
    # с.   -> село
    # гр.  -> град
    # м-т  -> местност
    # к.к.  -> курортен комплекс
    # anything else -> квартал/район
    .withColumn(
        "location_type",
        F.when(
            F.instr("location_text", "м-т ") > 0,
            "местност"
        )
        .when(
            F.instr("location_text", "с. ") > 0,
            "село"
        )
        .when(
            F.instr("location_text", "гр. ") > 0,
            "град"
        )
        .when(
            F.instr("location_text", "к.к. ") > 0,
            "курортен комплекс"
        )
        .otherwise("квартал/район")
    )

    # Extract the actual location name
    #
    # област Варна, с. Новаково
    # -> Новаково
    #
    # град Варна, м-т Долна Трака
    # -> Долна Трака
    .withColumn(
        "location_name",
        F.trim(
            F.regexp_replace(
                F.element_at(
                    F.split("location_text", ","),
                    -1
                ),
                r"^(с\.|гр\.|м-т)\s*",
                ""
            )
        )
    )

    # Extract the complete last-edited date/time only when
    # the expected Bulgarian format is present
    #
    # Example:
    # Коригирана в 7:51 на 24 август, 2026 год.
    .withColumn(
        "last_edited_match",
        F.regexp_extract(
            "listing_info_text",
            r"Коригирана в [0-9]{1,2}:[0-9]{2} на [0-9]{1,2} [^,]+, [0-9]{4} год\.",
            0
        )
    )

    # Extract time
    .withColumn(
        "last_edited_time",
        F.when(
            F.col("last_edited_match") != "",
            F.regexp_extract(
                "listing_info_text",
                r"Коригирана в ([0-9]{1,2}:[0-9]{2})",
                1
            )
        )
    )

    # Extract day
    .withColumn(
        "last_edited_day",
        F.when(
            F.col("last_edited_match") != "",
            F.regexp_extract(
                "listing_info_text",
                r"на ([0-9]{1,2}) [^,]+, [0-9]{4} год",
                1
            )
        )
    )

    # Extract Bulgarian month
    .withColumn(
        "last_edited_month",
        F.when(
            F.col("last_edited_match") != "",
            F.regexp_extract(
                "listing_info_text",
                r"на [0-9]{1,2} ([^,]+), [0-9]{4} год",
                1
            )
        )
    )

    # Extract year
    .withColumn(
        "last_edited_year",
        F.when(
            F.col("last_edited_match") != "",
            F.regexp_extract(
                "listing_info_text",
                r", ([0-9]{4}) год",
                1
            )
        )
    )

    # Convert Bulgarian month names to month numbers
    .withColumn(
        "last_edited_month_number",
        F.when(F.col("last_edited_month") == "януари", "01")
        .when(F.col("last_edited_month") == "февруари", "02")
        .when(F.col("last_edited_month") == "март", "03")
        .when(F.col("last_edited_month") == "април", "04")
        .when(F.col("last_edited_month") == "май", "05")
        .when(F.col("last_edited_month") == "юни", "06")
        .when(F.col("last_edited_month") == "юли", "07")
        .when(F.col("last_edited_month") == "август", "08")
        .when(F.col("last_edited_month") == "септември", "09")
        .when(F.col("last_edited_month") == "октомври", "10")
        .when(F.col("last_edited_month") == "ноември", "11")
        .when(F.col("last_edited_month") == "декември", "12")
    )

    # Build timestamp only when all required components exist
    .withColumn(
        "last_edited_at",
        F.when(
            (F.col("last_edited_year").isNotNull()) &
            (F.col("last_edited_month_number").isNotNull()) &
            (F.col("last_edited_day").isNotNull()) &
            (F.col("last_edited_time").isNotNull()),
            F.to_timestamp(
                F.concat(
                    F.col("last_edited_year"),
                    F.lit("-"),
                    F.col("last_edited_month_number"),
                    F.lit("-"),
                    F.lpad(F.col("last_edited_day"), 2, "0"),
                    F.lit(" "),
                    F.col("last_edited_time")
                ),
                "yyyy-MM-dd H:mm"
            )
        )
    )

    # Extract number of listing views
    #
    # Example:
    # Обявата е посетена 1471 пъти.
    .withColumn(
        "views",
        F.regexp_extract(
            "listing_info_text",
            r"посетена\s+([0-9]+)\s+пъти",
            1
        ).cast("long")
    )

    # Calculate price per square metre
    .withColumn(
        "price_per_sqm",
        F.when(
            (F.col("price").isNotNull()) &
            (F.col("area_sqm").isNotNull()) &
            (F.col("area_sqm") > 0),
            F.col("price") / F.col("area_sqm")
        )
    )

    # Remove temporary helper columns
    .drop(
        "area_raw",
        "plot_area_raw",
        "floor_raw",
        "gas_raw",
        "district_heating_raw",
        "construction_raw",
        "last_edited_match",
        "last_edited_time",
        "last_edited_day",
        "last_edited_month",
        "last_edited_year",
        "last_edited_month_number"
    )
)

# drop raw fields
silver_listings = silver_listings.drop(
    "url",
    "location_text",
    "listing_info_text",

)
# Inspect Silver

# display(
#     silver_listings.limit(50)
# )

# save silver 

silver_listings.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.silver_imot_listings")

spark.sql("""
SELECT *
FROM workspace.default.silver_imot_listings
LIMIT 100
""").display()

listing_id,agency_name,currency,price,description,feature_tags,area_sqm,plot_area_sqm,floor,has_gas,has_district_heating,district_heating_type,construction_type,region,location_type,location_name,last_edited_at,views,price_per_sqm
1j172561998634146,МД ОБЗОР ЕООД,EUR,249900.0,"ЕКСКЛУЗИВНА ОФЕРТА! БЕЗ КОМИСИОННА ОТ КУПУВАЧА! Имаме удоволствието да Ви предложим за продажба новоизградена луксозна двуетажна къща Американска мечта в град Бяла, която си заслужава да бъде разгледана! Разположена е в центъра на красивия морски курорт с лице към реновирана улица, в близост до новата автогара, магазини, училище. Къщата е с акт 14 от 2024г. и се продава на тапа в перфектно състояние и готова да посрещне новите си собственици, които да превърнат сградата в собствен уютен дом според своите предпочитания! Изпълнени: луксозна външна дограма, електро и ВиК инсталации, мазилка, замазка, инсталация за отопление и охлаждане за термопомпа с изводи за конвертори, автоматична гаражна врата, окабеляване за интернет и телевизия; излят фундамент за плътна ограда на парцела. В северната част на парцела има възможност новите собственици да изградят малък басейн, външен бар и други съоръжение за създаване на приятна атмосфера. Сградата е присъединена към водопроводната и канализационна система на града. Открита е партида за електричество. Перфектната къща за целогодишно обитаване. На новите собственици ще бъдат на разположение: Първи жилищен етаж с гараж на площ от 130.80кв.м., състоящ се от функционален гараж 25кв.м. със складово помещение /бойлерно/ 6.14кв.м. с директен достъп към дворното пространство и топла връзка към жилищната част, която е обособена като входно антре 7.50кв.м., стая за гости 10.10кв.м., баня 3.37кв.м. и просторна панорамна всекидневна /трапезария, кухня/ 48кв.м. с френски прозорци и директен достъп към покрита веранда и голям двор. Посредством вътрешно стълбище е осигурен достъп до Втори жилищен етаж 80кв.м. с разпределение: родителска спалня 16кв.м. със санитарен възел 5кв.м. и гардеробна 6кв.м. и огромна панорамна тераса; детска спалня с отделна баня; обособено е и перално помещение на площ от 4.40кв.м. Към момента се изпълнява външна изолация, след което цената ще се актуализира! Вашата мечта може да стане реалност!","List(Тухла, С гараж, С паркинг, Интернет връзка)",221.0,774.0,2,null,null,null,Тухла,Варна,град,гр. Бяла ЦЕНТЪР,2026-09-18T12:17:00.000Z,5468,1130.7692307692307
1j177124626112887,МОДА ИМОТИ,EUR,509999.0,"Стара цена: 550 000 / 1 075 706 лв. Нова цена: 509 999 / 997 472 лв. Отстъпка: 40 001 / 78 235 лв. Продава се самостоятелна къща в един от най-престижните райони на Варна кв. Св. Никола. Уникална локация с панорамна и напълно открита морска гледка, която не може да бъде закрита от бъдещо строителство. Застроена площ: 265 кв.м Парцел: 858 кв.м Имотът се намира на главната улица арх. Манол Йорданов, разделяща местностите Св. Никола и Долна Трака. Къщата е в отлично състояние, с функционално и удобно разпределение. Разпределение: Партер: Просторна дневна 55 кв.м с обособени кухненска, трапезарна и холна зона Санитарен възел Първи полуетаж: Спалня (подходяща за кабинет) Голяма гардеробна Баня с тоалетна Втори етаж: Две спални с морска панорама Баня с тоалетна Следващ полуетаж: Четвърта спалня със собствен санитарен възел Покривен етаж: Затворена зона с PVC дограма, предназначена за релакс Невероятна, напълно открита морска панорама Предимства: Топ локация в една от най-предпочитаните вилни зони на Варна Редовни документи Възможност за незабавно ползване Голям терасиран парцел с аранжирана растителност Обособени зони за стопанска дейност 4 паркоместа Къщата се продава обзаведена Панорамна къща с морска гледка на изключително атрактивна цена. Защо да изберете нас? 20 години опит на пазара на недвижими имоти; качество на обслужването и честност; ЛИЦЕНЗИРАН БРОКЕР нашият управител има магистърска степен по 'Икономика на недвижимите имоти' и е изучавал вещно право в университета, както и 'Основи на правото' в гимназията в с

# Bronze → Silver transformation (price history)

- Removed the url column because it isn't needed for price-history analysis.
- Converted price from its raw value into a numeric double.
- Converted observed_at from text into a proper timestamp.

In [0]:
from pyspark.sql import functions as F

# Create Silver price history without the URL column

silver_price_history = (
    bronze_price_history
    .select(
        "listing_id",
        F.expr("try_cast(price AS double)").alias("price"),
        "currency",
        F.to_timestamp("observed_at").alias("observed_at")
    )
)

# Overwrite the Silver table with the new schema

silver_price_history.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.default.silver_imot_price_history")


# Verify the data

spark.sql("""
SELECT *
FROM workspace.default.silver_imot_price_history
LIMIT 10
""").display()

listing_id,price,currency,observed_at
1j177705329084413,47400.0,EUR,2026-09-22T08:52:53.000Z
1j176303859226597,400000.0,EUR,2026-09-22T08:52:55.000Z
1j175870229657537,175000.0,EUR,2026-09-22T08:52:57.000Z
1j177643373590240,56000.0,EUR,2026-09-22T08:52:59.000Z
1j172043462655824,110000.0,EUR,2026-09-22T08:53:01.000Z
1j176520005776746,83000.0,EUR,2026-09-22T08:53:03.000Z
1j176607570585811,225000.0,EUR,2026-09-22T08:53:05.000Z
1j174585986671309,1000000.0,EUR,2026-09-22T08:53:07.000Z
1j176795909353231,418000.0,EUR,2026-09-22T08:53:10.000Z
1j175863968683541,1900000.0,EUR,2026-09-22T08:53:12.000Z


# Silver → Gold Transformation 

### The Silver layer contains cleaned property and price-history data, while the Gold layer contains aggregated market statistics, agency statistics, and price-change metrics.

## Source Silver Tables

The Gold transformations use two Silver tables:

- `silver_imot_listings`
  - Contains the current cleaned listing data.
  - Includes price, location, property characteristics, agency,
    views, area, and price per square metre.

- `silver_imot_price_history`
  - Contains historical price observations for each listing.
  - Includes listing ID, price, currency, and observation timestamp.

## Gold Tables Created

### 1. `gold_market_by_location`

This table provides aggregated market statistics by location.

The data is grouped by:

- Region
- Location name
- Location type
- Construction type

The table calculates:

- Number of listings
- Average price
- Median price
- Average price per square metre
- Median price per square metre
- Average property area

This table is intended to be the main source for market analysis
and Power BI dashboard visualisations.

### 2. `gold_agency_summary`

This table provides summary statistics for real-estate agencies.

The data is grouped by agency and calculates:

- Number of listings
- Average listing price
- Average listing views

This table can be used to analyse agency activity and create
agency-focused Power BI visualisations.

### 3. `gold_price_changes`

This table summarises price movement for each listing using the
historical price observations.

For each listing, it calculates:

- First observed price
- Latest observed price
- Absolute price change
- Percentage price change
- First observation date
- Latest observation date
- Number of price observations
- Number of days the listing has been tracked

The table is designed to become more useful over time as additional
price observations are collected by the scraper.

## Gold Layer Purpose

The Gold layer provides business-ready data instead of raw or
technical data structures.


In [0]:
from pyspark.sql import functions as F

silver_listings = spark.table("workspace.default.silver_imot_listings")
silver_price_history = spark.table("workspace.default.silver_imot_price_history")

# 1. Market summary by location 

gold_market_by_location = (
    silver_listings
    .filter(F.col("price").isNotNull() & F.col("price_per_sqm").isNotNull())
    .groupBy("region", "location_name", "location_type", "construction_type")
    .agg(
        F.count("*").alias("listing_count"),
        F.round(F.avg("price"), 0).alias("avg_price"),
        F.round(F.expr("percentile_approx(price, 0.5)"), 0).alias("median_price"),
        F.round(F.avg("price_per_sqm"), 2).alias("avg_price_per_sqm"),
        F.round(F.expr("percentile_approx(price_per_sqm, 0.5)"), 2).alias("median_price_per_sqm"),
        F.round(F.avg("area_sqm"), 1).alias("avg_area_sqm"),
    )
    .orderBy(F.desc("listing_count"))
)

gold_market_by_location.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.gold_market_by_location")


# 2. Agency summary 

gold_agency_summary = (
    silver_listings
    .filter(F.col("agency_name").isNotNull() & F.col("price").isNotNull())
    .groupBy("agency_name")
    .agg(
        F.count("*").alias("listing_count"),
        F.round(F.avg("price"), 0).alias("avg_price"),
        F.round(F.avg("views"), 0).alias("avg_views"),
    )
    .orderBy(F.desc("listing_count"))
)

gold_agency_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.gold_agency_summary")


# 3. Price changes per listing (self-populating as you scrape more days) 

price_window_stats = (
    silver_price_history
    .groupBy("listing_id")
    .agg(
        F.min("observed_at").alias("first_observed_at"),
        F.max("observed_at").alias("last_observed_at"),
        F.count("*").alias("observation_count"),
    )
)

first_price = (
    silver_price_history
    .withColumn(
        "rn_first",
        F.row_number().over(
            __import__("pyspark.sql.window", fromlist=["Window"]).Window
            .partitionBy("listing_id").orderBy("observed_at")
        )
    )
    .filter(F.col("rn_first") == 1)
    .select("listing_id", F.col("price").alias("first_price"))
)

latest_price = (
    silver_price_history
    .withColumn(
        "rn_last",
        F.row_number().over(
            __import__("pyspark.sql.window", fromlist=["Window"]).Window
            .partitionBy("listing_id").orderBy(F.desc("observed_at"))
        )
    )
    .filter(F.col("rn_last") == 1)
    .select("listing_id", F.col("price").alias("latest_price"))
)

gold_price_changes = (
    price_window_stats
    .join(first_price, "listing_id")
    .join(latest_price, "listing_id")
    .withColumn("price_change_abs", F.col("latest_price") - F.col("first_price"))
    .withColumn(
        "price_change_pct",
        F.when(
            F.col("first_price") > 0,
            F.round((F.col("latest_price") - F.col("first_price")) / F.col("first_price") * 100, 2)
        )
    )
    .withColumn(
        "days_tracked",
        F.datediff(F.col("last_observed_at"), F.col("first_observed_at"))
    )
)

gold_price_changes.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.gold_price_changes")

# Test: query gold_market_by_location

In [0]:
spark.sql("""
SELECT *
FROM workspace.default.gold_market_by_location
ORDER BY listing_count DESC
LIMIT 100
""").display()

region,location_name,location_type,construction_type,listing_count,avg_price,median_price,avg_price_per_sqm,median_price_per_sqm,avg_area_sqm
Варна,м-т Ален мак,местност,Тухла,129,563385.0,495000.0,2234.01,2112.68,261.3
Варна,м-т Горна Трака,местност,Тухла,105,501549.0,435000.0,2005.77,2004.61,264.0
Варна,м-т Долна Трака,местност,Тухла,86,629674.0,485000.0,2049.01,1980.0,314.3
Варна,Виница,квартал/район,Тухла,71,415260.0,399000.0,1870.81,1785.0,247.5
Варна,м-т Манастирски рид,местност,Тухла,67,470325.0,319000.0,1758.75,1687.5,260.5
Варна,с. Приселци,село,Тухла,67,331127.0,265000.0,1572.76,1575.76,246.2
Варна,гр. Бяла,град,Тухла,66,235137.0,235000.0,1284.77,1343.75,204.4
Варна,м-т Акчелар,местност,Тухла,64,506406.0,450000.0,2283.19,2111.54,232.3
Варна,м-т Боровец - юг,местност,Тухла,55,271493.0,260000.0,1566.58,1511.18,195.4
Варна,м-т Евксиноград,местност,Тухла,46,736891.0,570000.0,1985.85,2000.0,400.1


# Test: query gold_agency_summary

In [0]:
spark.sql("""
SELECT *
FROM workspace.default.gold_agency_summary
ORDER BY listing_count DESC
LIMIT 50
""").display()

agency_name,listing_count,avg_price,avg_views
ИМОТИ ПРЕМИЕР,148,306508.0,1613.0
АДРЕС,134,412342.0,1734.0
Частно лице,92,266694.0,1717.0
ЯВЛЕНА - ОФИС ВАРНА,81,318620.0,834.0
ХОЛДИНГ ГРУП,72,504136.0,1829.0
HOME2U,58,236392.0,1362.0
БЪЛГЕРИАН ПРОПЕРТИС,49,185421.0,2389.0
КУПИ В БЪЛГАРИЯ,49,357698.0,2329.0
ИМОТЕКА,48,343916.0,1742.0
ЕС ПРОПЪРТИС,43,461433.0,1464.0


# test: query gold_price_changes

### this will be useful after several scrapings and wil catch anyy price changes

<pre>
scraper runs
   ↓
current price compared with previous price
   ↓
price changed?
   ├─ NO  → don't add a history record
   └─ YES → add new record to price_history.jsonl
                    ↓
             Silver price history
                    ↓
             Gold price changes
             </pre>


In [0]:
spark.sql("""
SELECT *
FROM workspace.default.gold_price_changes
ORDER BY price_change_pct ASC
LIMIT 100
""").display()

spark.sql("""
SELECT *
FROM workspace.default.gold_price_changes
WHERE observation_count > 1
ORDER BY price_change_pct ASC
LIMIT 100
""").display()

listing_id,first_observed_at,last_observed_at,observation_count,first_price,latest_price,price_change_abs,price_change_pct,days_tracked
1j176045409122038,2026-09-22T09:29:44.000Z,2026-09-22T09:29:44.000Z,1,null,null,null,null,0
1j178471331186286,2026-09-22T09:24:16.000Z,2026-09-22T09:24:16.000Z,1,null,null,null,null,0
1j169045874863885,2026-09-22T09:35:53.000Z,2026-09-22T09:35:53.000Z,1,null,null,null,null,0
1j157030120325890,2026-09-22T10:08:05.000Z,2026-09-22T10:08:05.000Z,1,null,null,null,null,0
1j174946651338857,2026-09-22T09:10:34.000Z,2026-09-22T09:10:34.000Z,1,null,null,null,null,0
1j176397837416234,2026-09-22T10:08:29.000Z,2026-09-22T10:08:29.000Z,1,null,null,null,null,0
1j178428324225626,2026-09-22T09:25:02.000Z,2026-09-22T09:25:02.000Z,1,null,null,null,null,0
1j177797748066903,2026-09-22T08:57:08.000Z,2026-09-22T08:57:08.000Z,1,null,null,null,null,0
1j176650035582586,2026-09-22T09:33:24.000Z,2026-09-22T09:33:24.000Z,1,null,null,null,null,0
1j176355964139597,2026-09-22T09:03:10.000Z,2026-09-22T09:03:10.000Z,1,null,null,null,null,0


listing_id,first_observed_at,last_observed_at,observation_count,first_price,latest_price,price_change_abs,price_change_pct,days_tracked
1j174463637775440,2026-09-22T10:09:45.000Z,2026-09-24T15:02:34.000Z,2,null,420000.0,null,null,2
1j178885138054600,2026-09-22T09:10:12.000Z,2026-09-24T15:28:35.000Z,2,18000.0,14000.0,-4000.0,-22.22,2
1j174299821218339,2026-09-22T09:48:47.000Z,2026-09-24T14:39:03.000Z,2,65000.0,55000.0,-10000.0,-15.38,2
1j177986597708369,2026-09-22T09:52:08.000Z,2026-09-24T15:34:43.000Z,2,519900.0,459000.0,-60900.0,-11.71,2
1j176517805530669,2026-09-22T09:06:50.000Z,2026-09-24T15:25:47.000Z,2,500000.0,450000.0,-50000.0,-10.0,2
1j176518841903919,2026-09-22T09:06:52.000Z,2026-09-24T15:25:49.000Z,2,500000.0,450000.0,-50000.0,-10.0,2
1j168804487774584,2026-09-22T09:37:22.000Z,2026-09-24T14:28:28.000Z,2,31000.0,28000.0,-3000.0,-9.68,2
1j174254063475807,2026-09-22T09:42:48.000Z,2026-09-24T14:33:37.000Z,2,350000.0,320000.0,-30000.0,-8.57,2
1j178237494501382,2026-09-22T09:06:13.000Z,2026-09-24T15:25:12.000Z,2,175000.0,165000.0,-10000.0,-5.71,2
1j177563745209787,2026-09-22T09:21:48.000Z,2026-09-24T14:48:03.000Z,2,850000.0,815000.0,-35000.0,-4.12,2


download csv

In [0]:
gold_df = spark.table("workspace.default.gold_market_by_location")

gold_df.coalesce(1).write \
    .mode("overwrite") \
    .option("header", "true") \
    .csv("/Volumes/workspace/default/real-estate/gold_market_by_location_csv")

In [0]:
gold_df = spark.table("workspace.default.gold_agency_summary")

gold_df.coalesce(1).write \
    .mode("overwrite") \
    .option("header", "true") \
    .csv("/Volumes/workspace/default/real-estate/gold_agency_summary_csv")